# Introduction OLLAMA & LiteLLM et Introduction au AgentAI

Préparé avec ❤️ par <a href="https://linkedin.com/in/dany-anderson-guimefack">Dany Anderson G.</a> et <a href="https://www.linkedin.com/in/ariste-yougbar%C3%A9-735387332/">Armel  Ariste Y.</a> <br>
Email📧: dany.guimefack@centrale-casablanca.ma // wendenkomarmelariste.yougbare2@centrale-casablanca.ma




**Dans ce workshop**, nous allons :
1. Utiliser l'API **local/Ollama** pour des appels en local
2. Exploiter LiteLLM comme interface unifie de communication avec les differents modeles
3. Creer un systeme agentique pour la gestion locale des modeles avec Open-WebUI
4. Explorer quelques outils pour faciliter la programmation assité par IA

---

## Partie 1 Ollama : Configuration et Premier Appel

## Qu'est-ce que c'est que Ollama  ?

**Ollama** c’est un outil qui permet d’exécuter des modèles d’intelligence artificielle (LLM) directement sur son ordinateur, en local, sans passer par Internet.
On peut exécuter des modèles comme Llama, Mistral ou Gemma en local, avec une API local (http://localhost:11434) simple à utiliser.

## Pourquoi utiliser Ollama?
Ollama est interessant pour plusieurs raisons :

- exécuter des modèles localement ;
- tester rapidement plusieurs modèles ;
- éviter d'envoyer les données vers un service cloud ;
- prototyper des applications IA ;
- connecter facilement un modèle local à une API FastAPI ;
- travailler même sans compte OpenAI, Anthropic ou autre fournisseur.


In [ ]:
# Installation des bibliothèques nécessaires

%pip install ollama
%pip install dotenv

### Commandes importantes Ollama
Les commandes principales sont:
 - ` 1. ollama --version` # vérifier qu'Ollama est installer
 - ` 2. ollama list`      # voir les modèles installés
 - ` 3. ollama pull nom_du_modèle` # télécharger un modèle
 - ` 4. ollama run nom_du_modèle` # lancer un modèle
 - ` 5. ollama ps` # voir les modèles actuellement en cours d'exécution
 - ` 6. ollama stop nom_du_modèle` # arrêter un modèle
 - ` 7. ollama run nom_du_modèle` # supprimer un modèle

### Les Modèles à télécharger

     Exemple de Modèles:

| Modèle | Notes|
|---|---|
| `gemma3:1b` | `Fast responses, great for prototyping` |
| `llama3.1:8b` | `Good general-purpose modèle` |
| `qwen2.5:latest` | `Strong multilingual support` |
| `qwen2.5-coder:32b`| `Best for code generation` |
| `deepseek-r1:latest`| `Reasoning model` |
| `deepseek-coder-v2:latest`| `Code-focused` |



In [ ]:
# Telecharger votre premier Model
#CMD
#ollama pull gemma3:1b

# run votre modele sans api local
#ollama run gemma3:1b


### Discuter avec le Model


In [ ]:
#Sans API local

from ollama import chat

conversation=chat(
  model='gemma3:1b',
  messages=[{'role': 'user', 'content': 'combien font la racine cubique de 8'}],
  stream=True, # avoir les reponses en temps réels (en petit morceau)

)
in_thinking=False
for mot in conversation:
    "raisonnement interne"
    if mot.message.thinking:
        print(f'{mot.message.thinking}', end='', flush=True)#affiche le raisonnement en temps réel
    else:
        print(f'{mot.message.content}', end='', flush=True) #affiche la réponse en temps réel


In [ ]:
#Avec API local localhost:11434

import requests
import json

OLLAMA_API_URL = "http://localhost:11434"
Model_name = "gemma3:1b"
def converser_avec_modele(messages):
    conversation = {
        "model": Model_name,
        "messages": [{'role': 'user', 'content': messages } ],
        "stream": False
    }

    reponse = requests.post(f'{OLLAMA_API_URL}/api/chat', json=conversation)
    reponse.raise_for_status()
    data=reponse.json()
    return data["message"]["content"]

La racine cubique de 8 est 2, poichéde de 2.

En mathématiques, la racine cubique de un nombre est le nombre qui, multiplié par lui-même trois fois, donne ce nombre. Donc, 2 * 2 * 2 = 8.


---

## Partie 2 Hyperparamètres et Prompt Système  (Rappel)

### Quels hyperparamètres peut-on régler ?

| Hyperparamètre | Rôle | Valeur typique |
|---|---|---|
| `temperature` | Créativité vs déterminisme. `0` = réponse prévisible, `1` = très créatif | `0.7` – `1.0` |
| `max_output_tokens` | Longueur maximale de la réponse en tokens | `256` – `2048` |
| `top_p` | *Nucleus sampling* : ne garde que les tokens dont la probabilité cumulée dépasse ce seuil | `0.9` – `0.95` |
| `top_k` | Nombre de tokens candidats pris en compte à chaque étape | `40` – `100` |

### Le Prompt Système

Le **prompt système** (`system_instruction`) est une instruction globale, invisible pour l'utilisateur final, qui définit le comportement global du modèle :

- Son rôle (`"Tu es un expert en finance..."`)  
- Son ton (formel, décontracté, technique)  
- Ses contraintes (`"Réponds uniquement en français"`)  
- Le contexte dans lequel il opère (article, base de données, persona)

```
┌──────────────────────────────────────────────────────┐
│  System Instruction  →  "Tu es un assistant..."      │  ← invisible pour l'utilisateur
├──────────────────────────────────────────────────────┤
│  Message Utilisateur →  "Quelle est la capitale..."  │  ← saisie de l'utilisateur
└──────────────────────────────────────────────────────┘
```

---

## Partie 3 — Exemple Pratique : Questions sur un Article

Nous allons utiliser le **prompt système** pour fournir un article au modèle, puis créer une fonction qui répond aux questions en se basant **uniquement** sur cet article.

C'est une application concrète du pattern **RAG simplifié** (Retrieval-Augmented Generation) : on injecte un contexte dans le prompt système, et le modèle répond à partir de ce contexte uniquement.

In [ ]:
from ollama import chat

ARTICLE = """
Anthropic a conclu un partenariat stratégique avec SpaceX afin d'utiliser l'ensemble de la capacité
informatique du centre de données Colossus 1, situé à Memphis dans le Tennessee. Grâce à cet accord,
l'entreprise spécialisée dans l'intelligence artificielle bénéficiera de plus de 300 mégawatts de
puissance de calcul. Le groupe a également indiqué envisager une collaboration future avec SpaceX
autour du développement d'infrastructures de plusieurs gigawatts dans l'espace.

L'accord doit permettre à Anthropic d'améliorer les performances proposées aux abonnés de ses offres
Claude Pro et Claude Max. Cette alliance intervient pourtant dans un contexte de tensions publiques
entre Anthropic et Elon Musk, propriétaire de SpaceX et fondateur de xAI. Ces derniers mois, le
milliardaire a multiplié les critiques contre Anthropic, notamment sur les questions de régulation de
l'intelligence artificielle, allant jusqu'à accuser l'entreprise de \"détester la civilisation occidentale\".

Parallèlement, xAI poursuit l'expansion rapide de ses infrastructures à Memphis afin de concurrencer
OpenAI, Google et Anthropic sur le marché de l'IA. Le centre Colossus 1 est actuellement son plus
important site de calcul. Le projet fait toutefois face à des critiques locales liées à l'utilisation
de turbines au gaz naturel, accusées d'aggraver la pollution atmosphérique dans la région. Par ailleurs,
la startup fondée en 2021 par d'anciens chercheurs et dirigeants d'OpenAI serait en discussion avec
des investisseurs pour une levée de fonds susceptible de valoriser l'entreprise jusqu'à 900 milliards
de dollars.
"""


def repond_a_parti_de_larticle(text: str) -> str:
    """Répond à une question en se basant uniquement sur l'article fourni."""
    conversation=chat(
    model='gemma3:1b',
    messages=[{'role': 'system', 'content': "Tu es un assistant qui répond aux questions en te basant UNIQUEMENT sur l'article "
                "suivant. Si la réponse ne figure pas dans l'article, dis-le clairement.\n\n"
                f"Article :\n{ARTICLE}"}],
    stream=True, # avoir les reponses en temps réels (en petit morceau)

    )
    in_thinking=False
    for mot in conversation:
        "raisonnement interne"
        if mot.message.thinking:
            print(f'{mot.message.thinking}', end='', flush=True)#affiche le raisonnement en temps réel
        else:
            print(f'{mot.message.content}', end='', flush=True) #affiche la réponse en temps réel

    return mot.message.content

In [ ]:
# Test de la fonction avec plusieurs questions
questions = [
    "Quel est l'accord conclu entre Anthropic et SpaceX ?",
    "Quelles tensions sont mentionnées dans l'article ?",
    "À combien est valorisée l'entreprise dans les discussions de levée de fonds ?",
    "Quelle est la couleur préférée d'Elon Musk ?",  # hors article
]

for question in questions:
    print(f"Question : {question}")
    print(f"Réponse  : {repond_a_parti_de_larticle(question)}")
    print("-" * 70)

Question : Quel est l'accord conclu entre Anthropic et SpaceX ?
L'accord entre Anthropic et SpaceX consiste à utiliser la capacité informatique du centre de données Colossus 1, situé à Memphis, dans le Tennessee, de plus de 300 mégawatts de puissance de calcul. L’entreprise envisage également une collaboration future avec SpaceX pour le développement d’infrastructures de plusieurs gigawatts dans l’espace.Réponse  : 
----------------------------------------------------------------------
Question : Quelles tensions sont mentionnées dans l'article ?
L'accord entre Anthropic et SpaceX permettra à Anthropic d'améliorer les performances de ses offres Claude Pro et Claude Max grâce à plus de 300 mégawatts de puissance de calcul. L'entreprise envisage également une collaboration future avec SpaceX pour développer des infrastructures de plusieurs gigawatts dans l'espace.Réponse  : 
----------------------------------------------------------------------
Question : À combien est valorisée l'entrep

---


## Partie 4 : LiteLLM

Alors LiteLLM qu'est ce que c'est?.

### LiteLLM
LiteLLM est une couche d’abstraction pour appeler plusieurs modèles avec une API unifiée.

En effet :  sans LiteLLM , on a pour les differents fournisseurs une syntaxe propre à eux :

| Modèles | Notes |
|---|---|
| OpenAI | `from openai import OpenAI; client = OpenAI(api_key="KEY"); client.chat.completions.create()` |
| Anthropic | `import anthropic; client = anthropic.Anthropic(api_key="KEY"); client.messages.create()` |
| Ollama | `from ollama import chat; chat(model="llama3")` |
| OpenRouter | `from openai import OpenAI; client = OpenAI(api_key="KEY", base_url="https://openrouter.ai/api/v1"); client.chat.completions.create()` |
| Azure | `from openai import AzureOpenAI; client = AzureOpenAI(api_key="KEY", azure_endpoint="URL", api_version="2024-02-15-preview"); client.chat.completions.create()` |


Avec LiteLLM : [voir Documentation](https://docs.litellm.ai/docs/)
- On garde le même format d'appel
- faciliter pour changer les modèles



In [ ]:
#Installation de LiteLLM
#%pip install litellm=1.83.1

In [ ]:
#EXEMPLE AVEC Ollama et structure de redaction et de sortie en JSON

from littellm import completion


response = completion(
    model="ollama/qwen3",
    messages=[
        {"role": "user", "content": "Explique ce qu'est un LLM simplement."}
    ],
    api_base="http://localhost:11434"
)

print(response.choices[0].message.content)


#Format de Reponse

# {
#   "id": "chatcmpl-abc123",
#   "object": "chat.completion",
#   "created": 1677858242,
#   "model": "gpt-4o",
#   "choices": [
#     {
#       "index": 0,
#       "message": {
#         "role": "assistant",
#         "content": "Hello! I'm doing well, thanks for asking."
#       },
#       "finish_reason": "stop"
#     }
#   ],
#   "usage": {
#     "prompt_tokens": 13,
#     "completion_tokens": 12,
#     "total_tokens": 25
#   }
# }

---

## Partie  — Application Web Flask avec un Modèle Local

Nous allons créer une interface web pour interagir avec un modèle IA hébergé **localement** sur votre machine, sans passer par une API externe.

### Structure du projet Flask

```
chat_interface_fastapi/
├── main.py
├── requirements.txt
├── README.md
├── static/
│   └── style.css
├── templates/
│   └── index.html
└── assets/
    └── .gitkeep
```



### Lancer l'application

```bash
cd chat_interface_fastapi
pip install -r requirements.txt
uvicorn main:app --reload    
```

Puis ouvrez [http://127.0.0.1:8000](http://127.0.0.1:8000) dans votre navigateur.

---

## Partie 3 — Creation d'un Mini AgentIA






In [5]:
import requests

response = requests.get(
    "http://localhost:11434",)
print(response.text)

Ollama is running
